In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import pandas as pd

df = pd.read_csv("/content/cancer patient data sets.csv")
df.head()


,index,Patient Id,Age,Gender,Air Pollution,Alcohol use,Dust Allergy,OccuPational Hazards,Genetic Risk,chronic Lung Disease,...,Fatigue,Weight Loss,Shortness of Breath,Wheezing,Swallowing Difficulty,Clubbing of Finger Nails,Frequent Cold,Dry Cough,Snoring,Level
0,0,P1,33,1,2,4,5,4,3,2,...,3,4,2,2,3,1,2,3,4,Low
1,1,P10,17,1,3,1,5,3,4,2,...,1,3,7,8,6,2,1,7,2,Medium
2,2,P100,35,1,4,5,6,5,5,4,...,8,7,9,2,1,4,6,7,2,High
3,3,P1000,37,1,7,7,7,7,6,7,...,4,2,3,1,4,5,6,7,5,High
4,4,P101,46,1,6,8,7,7,7,6,...,3,2,4,1,4,2,4,2,3,High


In [ ]:
df.columns

Index(['index', 'Patient Id', 'Age', 'Gender', 'Air Pollution', 'Alcohol use',
       'Dust Allergy', 'OccuPational Hazards', 'Genetic Risk',
       'chronic Lung Disease', 'Balanced Diet', 'Obesity', 'Smoking',
       'Passive Smoker', 'Chest Pain', 'Coughing of Blood', 'Fatigue',
       'Weight Loss', 'Shortness of Breath', 'Wheezing',
       'Swallowing Difficulty', 'Clubbing of Finger Nails', 'Frequent Cold',
       'Dry Cough', 'Snoring', 'Level', 'text'],
      dtype='object')

In [ ]:
# เลือกเฉพาะ column ที่จำเป็น
df = df[["text", "Level"]]
df.head()

,text,Level
0,index is 0. Patient Id is P1. Age is 33. Gende...,Low
1,index is 1. Patient Id is P10. Age is 17. Gend...,Medium
2,index is 2. Patient Id is P100. Age is 35. Gen...,High
3,index is 3. Patient Id is P1000. Age is 37. Ge...,High
4,index is 4. Patient Id is P101. Age is 46. Gen...,High


In [ ]:
df = df.rename(columns={"Level": "label"})
df.head()

,text,label
0,index is 0. Patient Id is P1. Age is 33. Gende...,Low
1,index is 1. Patient Id is P10. Age is 17. Gend...,Medium
2,index is 2. Patient Id is P100. Age is 35. Gen...,High
3,index is 3. Patient Id is P1000. Age is 37. Ge...,High
4,index is 4. Patient Id is P101. Age is 46. Gen...,High


In [ ]:
df["label"].unique()

array(['Low', 'Medium', 'High'], dtype=object)

In [ ]:
label_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

df["label"] = df["label"].map(label_map)
df.head()

,text,label
0,index is 0. Patient Id is P1. Age is 33. Gende...,0
1,index is 1. Patient Id is P10. Age is 17. Gend...,1
2,index is 2. Patient Id is P100. Age is 35. Gen...,2
3,index is 3. Patient Id is P1000. Age is 37. Ge...,2
4,index is 4. Patient Id is P101. Age is 46. Gen...,2


In [ ]:
# แปลงเป็น Dataset ของ Hugging Face
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})

In [ ]:
dataset = dataset.train_test_split(test_size=0.2)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 800
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
})

In [ ]:
# แปลง  text เป็นตัวเลข
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize, batched=True)


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
    save_total_limit=1,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer
)

trainer.train()

/tmp/ipython-input-1153156244.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,No log,0.550735
2,No log,0.143953
3,No log,0.096164


TrainOutput(global_step=150, training_loss=0.4871436564127604, metrics={'train_runtime': 34.4148, 'train_samples_per_second': 69.737, 'train_steps_per_second': 4.359, 'total_flos': 79481856614400.0, 'train_loss': 0.4871436564127604, 'epoch': 3.0})

In [ ]:
sample = "Age is 60. Smoking is high. Chronic lung disease present."
inputs = tokenizer(sample, return_tensors="pt")
# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}
outputs = model(**inputs)

prediction = outputs.logits.argmax(dim=1).item()

label_map_reverse = {0: "Low", 1: "Medium", 2: "High"}
print("Predicted Level:", label_map_reverse[prediction])

Predicted Level: High
